# H1 - Forme d'occurrence de *fairness* selon le type d'acteurice

Question : la forme sous laquelle *fairness* est employe depend-elle du type d'acteurice ?

On classe chaque occurrence dans une categorie de forme, lue sur l'ego-reseau AMR :

- a : invocation nue
- b : enumeration
- c : modifieur de domaine
- d : definition
- e : predicative (le terme porte un argument propre, *fairness of X* compris)
- mixte : l'occurrence remplit au moins deux des conditions b, c, d, e

puis on croise avec le type d'acteurice et on teste l'association.

Conventions decidees en amont :

- *fair-01* et *fairness* sont fusionnes en un seul concept ; la categorie se lit sur la structure, pas sur l'etiquette.
- l'unite d'analyse est l'occurrence (une ligne par noeud mot-cle).
- les occurrences sans type d'acteurice sont exclues du test.

Avertissement : ce notebook tourne sur un echantillon de test de 100 phrases, petit et desequilibre entre acteurices. Il valide la chaine de traitement et le test statistique ; il ne produit pas un resultat d'association publiable.

In [1]:
from pathlib import Path

DATA_DIR = Path(".")  # dossier contenant les donnees ; adapter a l'emplacement dans le depot
PENMAN_FILE = DATA_DIR / "fairness_sample_penmans.txt"
ACTOR_MAP   = DATA_DIR / "fairness_sample_doc_actor_map.csv"

# Concepts traites comme le mot-cle (fair-01 et fairness fusionnes)
KEYWORD_RE = r"^(fairness|fair-\d+)$"

N_PERMUTATIONS = 2000
RANDOM_SEED = 0

In [2]:
import re
import pandas as pd
import numpy as np
import penman
from scipy.stats import chi2_contingency

## Regle de classification a-e

Pour chaque occurrence on ne regarde que les aretes propres du noeud mot-cle (relations vers ses parents directs, relations vers ses enfants). Un noeud peut remplir plusieurs conditions ; dans ce cas il recoit la categorie *mixte*.

- e (predicative) : le noeud a un enfant argument (:ARG0..:ARG5).
- c (modifieur de domaine) : le noeud a un enfant :mod ou :domain.
- d (definition) : le parent direct est un predicat de definition (`DEFINITION_FRAMES`) et le terme y occupe la place :ARG0 ou :ARG1.
- b (enumeration) : le parent direct est une conjonction (and / or) reliee par :opN.
- a (invocation nue) : aucune des conditions ci-dessus.

`DEFINITION_FRAMES` et la frontiere entre categories sont des decisions d'analyste, editables ci-dessous.

In [3]:
# Predicats de definition (categorie d). Liste editable : decision d'analyste.
DEFINITION_FRAMES = {"mean-01", "refer-01", "imply-01", "define-01",
                     "denote-01", "describe-01", "constitute-01"}
DEFINIENDUM = {":ARG0", ":ARG1"}   # places du terme defini sous un predicat de definition
CAT_COL = "category"

In [4]:
keyword_re = re.compile(KEYWORD_RE)
CORE_ARGS = {f":ARG{i}" for i in range(6)}   # arguments de coeur :ARG0..:ARG5

def parse_metadata(block):
    head = "\n".join(l for l in block.splitlines() if l.startswith("#"))
    sid = re.search(r"::id (\S+)", head)
    did = re.search(r"::doc_id (\S+)", head)
    snt = re.search(r"::snt (.+)", head)
    return (sid.group(1) if sid else None,
            int(did.group(1)) if did else None,
            snt.group(1).strip() if snt else "")

def canonical_edges(g):
    # remet les aretes inverses (:role-of) dans le sens tete -> dependant
    edges = []
    for s, r, t in g.edges():
        if r.endswith("-of") and r != ":consist-of":
            edges.append((t, ":" + r[1:-3], s))
        else:
            edges.append((s, r, t))
    return edges

In [5]:
def classify_block(block):
    sid, did, snt = parse_metadata(block)
    graph_str = "\n".join(l for l in block.splitlines() if not l.startswith("#"))
    g = penman.decode(graph_str)
    concept = {v: c for v, _, c in g.instances()}
    E = canonical_edges(g)
    rows = []
    kw_vars = sorted(v for v, c in concept.items() if c and keyword_re.match(c))
    for i, v in enumerate(kw_vars):
        children = [(r, dep) for h, r, dep in E if h == v]
        parents  = [(h, r)   for h, r, dep in E if dep == v]
        has_e = any(r in CORE_ARGS for r, _ in children)
        has_c = any(r in (":mod", ":domain") for r, _ in children)
        has_d = any(concept.get(h) in DEFINITION_FRAMES and r in DEFINIENDUM
                    for h, r in parents)
        has_b = any(re.match(r":op\d+$", r) and concept.get(h) in ("and", "or")
                    for h, r in parents)
        flags = [k for k, on in (("b", has_b), ("c", has_c), ("d", has_d), ("e", has_e)) if on]
        category = "a" if not flags else (flags[0] if len(flags) == 1 else "mixte")
        rows.append(dict(sample_id=sid, doc_id=did, occ=i,
                         keyword_surface=concept[v], category=category,
                         flags="".join(flags), sentence=snt))
    return rows

In [6]:
raw = PENMAN_FILE.read_text(encoding="utf-8")
blocks = [b for b in raw.split("\n\n") if b.strip()]

occ_rows, no_keyword = [], 0
for b in blocks:
    r = classify_block(b)
    if not r:
        no_keyword += 1
    occ_rows += r

df = pd.DataFrame(occ_rows)
df["keyword_norm"] = "fairness"   # fusion fair-01 / fairness

print(f"phrases (blocs) : {len(blocks)}")
print(f"phrases sans noeud mot-cle : {no_keyword}")
print(f"occurrences trouvees : {len(df)}")
print("\netiquette de surface avant fusion :")
print(df["keyword_surface"].value_counts().to_string())
print("\ndistribution des categories de forme :")
print(df["category"].value_counts().to_string())

phrases (blocs) : 100
phrases sans noeud mot-cle : 3
occurrences trouvees : 106

etiquette de surface avant fusion :
keyword_surface
fairness    79
fair-01     27

distribution des categories de forme :
category
b        46
a        33
mixte    11
c         7
e         6
d         3


In [7]:
actor = pd.read_csv(ACTOR_MAP)[["doc_id", "actor_type", "needs_review"]]
df = df.merge(actor, on="doc_id", how="left")

n_nan = df["actor_type"].isna().sum()
print(f"occurrences sans acteurice (exclues du test) : {n_nan}")
print(f"documents acteurice marques needs_review : {int(actor['needs_review'].sum())} / {len(actor)} "
      "(etiquettes provisoires, non reverifiees)")

test_df = df.dropna(subset=["actor_type"]).copy()
print(f"occurrences retenues pour le test : {len(test_df)}")

occurrences sans acteurice (exclues du test) : 9
documents acteurice marques needs_review : 41 / 52 (etiquettes provisoires, non reverifiees)
occurrences retenues pour le test : 97


In [8]:
ct = pd.crosstab(test_df["category"], test_df["actor_type"])
ct

actor_type,academia,civil society,international organisation,national authority,private sector
category,,,,,
a,1,10,7,10,1
b,1,11,13,17,2
c,0,1,1,4,0
d,0,1,1,1,0
e,0,1,1,4,0
mixte,0,1,2,5,1


## Test d'association

- chi2 d'independance : compare la table observee a celle qu'on aurait si la categorie et l'acteurice etaient sans lien. La p-value asymptotique n'est fiable que si chaque valeur attendue depasse ~5.
- comme la table est creuse, on ajoute un test de permutation : on melange les etiquettes d'acteurice N fois, on recalcule le chi2, et la p-value est la fraction des tirages au moins aussi extremes que l'observe. C'est l'equivalent robuste du test exact pour une table plus grande que 2 x 2.
- V de Cramer : force du lien, entre 0 (aucun) et 1 (parfait). Reperes conventionnels : 0,1 faible, 0,3 moyen, 0,5 fort.

On lit ensemble la p-value (le lien est-il reel ou du au hasard) et le V (est-il assez fort pour compter).

In [9]:
chi2, p_asym, dof, expected = chi2_contingency(ct)
n = ct.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))

# test de permutation (robuste pour cases creuses)
rng = np.random.default_rng(RANDOM_SEED)
labels_cat = test_df[CAT_COL].to_numpy()
labels_act = test_df["actor_type"].to_numpy()
count = 0
for _ in range(N_PERMUTATIONS):
    perm = rng.permutation(labels_act)
    c, _, _, _ = chi2_contingency(pd.crosstab(pd.Series(labels_cat), pd.Series(perm)))
    if c >= chi2:
        count += 1
p_perm = (count + 1) / (N_PERMUTATIONS + 1)

print(f"chi2 = {chi2:.2f}   dof = {dof}")
print(f"plus petite valeur attendue = {expected.min():.2f}  (si < 5, se fier a la permutation)")
print(f"p asymptotique (chi2)      = {p_asym:.3f}")
print(f"p permutation (B={N_PERMUTATIONS}) = {p_perm:.3f}")
print(f"V de Cramer                = {cramers_v:.3f}")

chi2 = 8.02   dof = 20
plus petite valeur attendue = 0.06  (si < 5, se fier a la permutation)
p asymptotique (chi2)      = 0.992
p permutation (B=2000) = 0.987
V de Cramer                = 0.144


## Apercu pour inspecter la regle

A relire pour valider ou corriger le classement sur des cas reels.

In [10]:
pd.set_option("display.max_colwidth", 90)
test_df[["sample_id", "category", "flags", "sentence"]].head(20)

,sample_id,category,flags,sentence
0,s000,mixte,bd,The European social partners furthermore refer to principles of fairness (no unfair bi...
1,s001,d,d,"This highlights challenges in defining what we mean by fairness, which is a complex an..."
2,s002,b,b,"lawful, ethical and robust) AI: 1) Human agency an d oversight, 2)Technical robustness..."
3,s003,b,b,"For instance, how could AI systems be designed so they respect and promote human right..."
4,s004,d,d,Assessment List for Trustworthy AI (ALTAI) 17 • Is your definition of fairness commonl...
5,s005,b,b,"Or, under certain conditions such as data sparsity, privacy-enhancing techniques can r..."
6,s006,d,d,"Considering that several definitions of fairness exist, it is useful to understand how..."
7,s007,b,b,Organisations that are introducing algorithms into decisions that were previously pure...
8,s008,b,b,‘Algorithmic Transparency and Platform Loyalty or Fairness in the French Digital Repub...
9,s010,b,b,This includes fostering the understanding of AI systems more generally (including a ba...


## Limites de cet echantillon

- Desequilibre fort entre acteurices (academia et private sector quasi vides) : le test n'a presque aucune puissance pour ces deux types.
- Petit effectif et table tres creuse : la p-value asymptotique du chi2 est peu fiable, d'ou la permutation.
- Etiquettes d'acteurice provisoires (majorite marquee needs_review), non reverifiees.
- Les occurrences d'un meme document ne sont pas independantes ; pour le corpus complet, prevoir une verification de robustesse (ponderation ou regroupement par document).
- La liste DEFINITION_FRAMES et la frontiere entre categories sont des decisions d'analyste a documenter.

En resume : ce notebook montre que la chaine tourne de bout en bout et produit table, test et taille d'effet ; il ne fournit pas un resultat interpretable sur ce jeu de test.